# Task 01/02 -- Copula family & tail dependence (`docs/open_questions.md` §4.2, §8.3)

Measures real empirical upper-tail dependence for a PM-coupled pair
(`was`, `wsh`) against the Gaussian copula's predicted zero. Serves both
Task 01 (§4.2: which copula family) and Task 02 (§8.3: rejection sampling
vs. fixing the copula) -- `open_questions.md` states these are the same
underlying question from two angles. See `tasks/01-copula-family-tail-dependence.md`
and `tasks/02-rejection-sampling-vs-copula-fix.md`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorly as tl
import xarray as xr
from tensorly.decomposition import partial_tucker

from train.ingest import load_training_window

tl.set_backend("numpy")

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("../data")

# Window entirely inside era 3 (starts 2025-10-27, core.archive_eras.ARCHIVE_ERAS);
# load_training_window rejects any window spanning more than one era.
T_START = datetime(2025, 11, 1, tzinfo=timezone.utc)
T_END = datetime(2026, 2, 1, tzinfo=timezone.utc)

VARIABLES = ["was", "wsh"]  # wind speed (wind grid) / wind-sea height (wave grid)
X_raw = load_training_window(DATA_DIR, T_START, T_END, VARIABLES)

# Both grids are (721, 1440) with identical lat0/step in era 3 (core.config),
# so no reprojection is needed to index the same (lat, lon) cell across the two.
ocean_mask = (
    ~np.isnan(X_raw["was"]).any(dim="time").values
    & ~np.isnan(X_raw["wsh"]).any(dim="time").values
)
ocean_cells = np.argwhere(ocean_mask)
lat_idx, lon_idx = ocean_cells[len(ocean_cells) // 2]  # one representative ocean cell
print(f"{ocean_mask.sum()} shared ocean cells; using ({lat_idx}, {lon_idx})")


## Gaussianize both variables (today's default placement, per Task 00)

In [ ]:
from train.copula import GaussianCopula

gaussianized = GaussianCopula(n_bins=500).fit_transform(X_raw[VARIABLES])
z_was = gaussianized["was"].values[:, lat_idx, lon_idx]
z_wsh = gaussianized["wsh"].values[:, lat_idx, lon_idx]


## Empirical upper-tail dependence vs. the Gaussian copula's predicted 0

In [ ]:
from scipy.stats import norm

q = 0.95
u_was, u_wsh = norm.cdf(z_was), norm.cdf(z_wsh)
lambda_U_empirical = np.mean((u_was > q) & (u_wsh > q)) / (1 - q)
print(f"lambda_U (q={q}) at cell ({lat_idx},{lon_idx}): {lambda_U_empirical:.3f}")

# gaussian_copula_and_marginals.md §2.5: a Gaussian copula predicts
# lambda_U -> 0 as q -> 1 for any rho < 1. Repeat across several q and several
# cells/seasons before drawing a conclusion -- one cell is illustrative only.


## Task 02's framing: what fraction of real co-occurrences would a Gaussian-copula-implied joint miss?

Fit a Gaussian copula's implied bivariate normal (correlation from the
Gaussianized pair) and compare its predicted joint upper-tail probability
against the empirical one measured above -- the gap is a direct estimate of
how often rejection sampling would need to fire (Task 02) versus how much a
tail-dependent copula family would remove at the source (Task 01).

In [ ]:
rho = np.corrcoef(z_was, z_wsh)[0, 1]
from scipy.stats import multivariate_normal

mvn = multivariate_normal(mean=[0, 0], cov=[[1, rho], [rho, 1]])
z_q = norm.ppf(q)
p_joint_gaussian = mvn.cdf([100, 100]) - mvn.cdf([z_q, 100]) - mvn.cdf([100, z_q]) + mvn.cdf([z_q, z_q])
p_joint_empirical = np.mean((u_was > q) & (u_wsh > q))
print(f"P(both > q={q}): Gaussian-implied {p_joint_gaussian:.4f} vs. empirical {p_joint_empirical:.4f}")
